# Setup

In [ ]:
import os
import json

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "ibm-granite/granite-4.1-8b"

In [5]:
from openrouter import OpenRouter
import langchain_openrouter

from langchain_core.messages import HumanMessage
from langchain_core.messages import SystemMessage
from langchain_core.messages import AIMessage

from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser

from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.stores import InMemoryStore

from langchain_classic.chains import RetrievalQA
from langchain_classic.chains import ConversationChain
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SequentialChain

from langchain_classic.memory import ChatMessageHistory
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.memory import ConversationSummaryBufferMemory

from pydantic import BaseModel
from pydantic import Field

In [ ]:
## list all the parameters that can be used to create a chat model
help(langchain_openrouter.ChatOpenRouter)

# Create Model

In [7]:
def llm_model(params=None):

    # 1. Define sensible defaults
    config = {
        "model": OPENROUTER_MODEL,
        "api_key": OPENROUTER_API_KEY,
        "base_url": OPENROUTER_DEFAULT_BASE_URL,
        "temperature": 0.5,
        "max_tokens": 256,
        "max_completion_tokens": 128
    }
    
    if params:
        config.update(params)
        
    # 3. Initialize the model
    model = langchain_openrouter.ChatOpenRouter(
        model=config["model"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        max_completion_tokens=config["max_completion_tokens"]
    )

    return model

def llm_model_response(prompt_text, params=None):
            
    # 3. Initialize the model
    model = llm_model(params)

    response = model.invoke(prompt_text)

    return response

# Langchain Concepts

## Chat Message

In [ ]:
from prompt_toolkit import prompt


OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assists a user in choosing the perfect book to read in one short sentence"),
        HumanMessage(content="I enjoy mystery novels, what should I read?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)


In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

msg = model.invoke(
    [
        # can also exclude SystemMessage and it will default to a helpful assistant
        #SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)

print(json.dumps(msg.response_metadata, indent=4))
print(msg.content)

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

# creative
params_creative = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

# precise
params_precise = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,  
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.2,
    "max_tokens": 512,
    "max_completion_tokens": 256
}

model_creative = llm_model(params=params_creative)
model_precise = llm_model(params=params_precise)

prompts = [
    "Write a short poem about artificial intelligence",
    "What are the key components of a neural network?",
    "List 5 tips for effective time management"
]

for prompt in prompts:
    response_creative = model_creative.invoke(prompt)
    response_precise = model_precise.invoke(prompt)

    print(f"Prompt: {prompt}")
    print(f"Creative Response: {response_creative.text}")
    print(f"Precise Response: {response_precise.text}")
    print("-" * 50)



## Prompt Templates

### String prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

prompt = PromptTemplate.from_template("Tell me one {adjective} joke about {topic}")

input = {"adjective": "funny", "topic": "cats"} 

prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Chat prompt templates

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a list of message tuples
# Each tuple contains a role ("system" or "user") and the message content
# The system message sets the behavior of the assistant
# The user message includes a variable placeholder {topic} that will be replaced later
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Tell me a joke about {topic}")
])

# Create a dictionary with the variable to be inserted into the template
# The key "topic" matches the placeholder name in the user message
input = {"topic": "cats"}

# Format the chat template with our input values
# This replaces {topic} with "cats" in the user message
# The result will be a formatted chat message structure ready to be sent to a model
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

### Messages Placeholder

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a ChatPromptTemplate with a system message and a placeholder for multiple messages
# The system message sets the behavior for the assistant
# MessagesPlaceholder allows for inserting multiple messages at once into the template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("msgs")  
])

# Create an input dictionary where the key matches the MessagesPlaceholder name
# The value is a list of message objects that will replace the placeholder
# Here we're adding a single HumanMessage asking about the day after Tuesday
input = {"msgs": [HumanMessage(content="What is the day after Tuesday?")]}

# Format the chat template with our input dictionary
# This replaces the MessagesPlaceholder with the HumanMessage in our input
# The result will be a formatted chat structure with a system message and our human message
prompt.invoke(input)

chain = prompt | model
response = chain.invoke(input = input)

print(json.dumps(response.response_metadata, indent=4))
print(response.content)

## Output Parsers

### JSON parser

In [ ]:
# Define your desired data structure.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

joke_query = "Tell me a joke."

# Set up a parser + inject instructions into the prompt template.
output_parser = JsonOutputParser(pydantic_object=Joke)

# Get the formatting instructions for the output parser
# This generates guidance text that tells the LLM how to format its response
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that includes:
# 1. Instructions for the LLM to answer the user's query
# 2. Format instructions to ensure the LLM returns properly structured data
# 3. The actual user query placeholder
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"query": joke_query})

print(json.dumps(response, indent=4))

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = JsonOutputParser()

format_instructions = """RESPONSE FORMAT INSTRUCTIONS: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant. Task: Generate info about the movie "{movie_name}" in JSON format. {format_instructions}""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

movie_chain = prompt_template | model | output_parser
movie_name = "Inception"
response = movie_chain.invoke({"movie_name": movie_name})

print("Parsed result:")
print(f"Title: {response['title']}")
print(f"Director: {response['director']}")
print(f"Year: {response['year']}")
print(f"Genre: {response['genre']}")




### Comma-separated list parser

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create an instance of the parser that will convert comma-separated text into a Python list
output_parser = CommaSeparatedListOutputParser()

# Get formatting instructions that will tell the LLM how to structure its response
# These instructions explain to the LLM that it should return items in a comma-separated format
format_instructions = output_parser.get_format_instructions()

# Create a prompt template that:
# 1. Instructs the LLM to answer the user query
# 2. Includes format instructions so the LLM knows to respond with comma-separated values
# 3. Asks the LLM to list five items of the specified subject
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{subject}\n",
    input_variables=["subject"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

# Create a processing chain that:
# 1. Formats the prompt using the template
# 2. Sends the formatted prompt to the LLM
# 3. Parses the LLM's response using the output parser to extract structured data
chain = prompt | model | output_parser

# Invoke the chain with a specific query about jokes
# This will:
# 1. Format the prompt with the joke query
# 2. Send it to the LLM
# 3. Parse the response into the structure defined by your output parser
# 4. Return the structured result
response = chain.invoke({"subject": "ice cream flavors"})

print(response)

## Documents

### Document Object

In [ ]:
Document( 
    page_content="""Python is an interpreted high-level general-purpose programming language.
                    Python's design philosophy emphasizes code readability with its notable use of significant indentation.""",
    metadata= {
        'my_document_id' : 234234,                      # Unique identifier for this document
        'my_document_source' : "About Python",          # Source or title information
        'my_document_create_time' : 1680013019          # Unix timestamp for document creation (March 28, 2023)
    }
)

### Document Loaders

Document loaders in LangChain are designed to load documents from a variety of sources; for instance, loading a PDF file and having the LLM read the PDF file using LangChain.

LangChain offers over 100 distinct document loaders, along with integrations with other major providers, such as AirByte and Unstructured. These integrations enable loading of all kinds of documents (HTML, PDF, code) from various locations including private Amazon S3 buckets, as well as from public websites).

In [ ]:
## PDF Loader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

document = loader.load()

## print formatted json metadata of the first page of the document
print(json.dumps(document[0].metadata, indent=4))
print(document[0].page_content)
##print(document[0].page_content[:1000])


## Splitters
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))
print(chunks[0].page_content)


In [ ]:
### Websire Loader
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")

web_data = loader.load()

print(web_data[0].page_content[:1000])

## Splitters
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
texts = text_splitter.split_documents(web_data)
print(len(texts))

In [ ]:
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30, separators=["\n\n", "\n", " ", ""])

chunks_pdf = splitter_1.split_documents(pdf_document)
chunks_web = splitter_2.split_documents(web_document)

def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

display_document_stats(chunks_pdf, "PDF Document")
display_document_stats(chunks_web, "Web Document")


### Embedding

Embedding models are specifically designed to interface with text embeddings.

Embeddings generate a vector representation for a specified piece or "chunk" of text.  Embeddings offer the advantage of allowing you to conceptualize text within a vector space. Consequently, you can perform operations such as semantic search, where you identify pieces of text that are most similar within the vector space.


In [ ]:
OPENROUTER_MODEL = "openai/text-embedding-3-small"

## PDF Loader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

document = loader.load()

## print formatted json metadata of the first page of the document
##print(json.dumps(document[0].metadata, indent=4))
##print(document[0].page_content)
##print(document[0].page_content[:1000])


## Splitters
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100, separator="\n")
chunks = text_splitter.split_documents(document)
##print(len(chunks))
##print(chunks[0].page_content)

from openrouter import OpenRouter

texts = [chunk.page_content for chunk in chunks]

with OpenRouter(api_key=OPENROUTER_API_KEY, server_url=OPENROUTER_DEFAULT_BASE_URL) as openrouter:
    result = openrouter.embeddings.generate(
        model=OPENROUTER_MODEL,
        input=texts
    )
    
print(result.data[0].embedding[:100])  # Print the first 100 dimensions of the first embedding vector


### Embbeding, Batches

In [ ]:
# Usar OpenRouter e n\ao as libs de langchain, langchain_openrouter n~\ao suporta embbedings
#from openrouter import OpenRouter # Ensure this matches your local SDK package structure

OPENROUTER_MODEL = "openai/text-embedding-3-small"

# 1. Load document
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # Increased for better contextual meaning
    chunk_overlap=120,    # Generous overlap to keep context intact
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
texts = [chunk.page_content for chunk in chunks]

# 3. Batching API calls (Handling max 100 texts at a time)
batch_size = 100
all_embeddings = []

with OpenRouter(api_key=OPENROUTER_API_KEY, server_url=OPENROUTER_DEFAULT_BASE_URL) as openrouter:
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        result = openrouter.embeddings.generate(
            model=OPENROUTER_MODEL,
            input=batch,
            dimensions=1024
        )
        all_embeddings.extend([item.embedding for item in result.data])

print(f"Successfully generated {len(all_embeddings)} embeddings.")
print(f"Sample of first embedding vector: {all_embeddings[0][:5]}...")

In [ ]:
#usar langchain_openai

OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
texts = [chunk.page_content for chunk in chunks]

# 3. Native LangChain Embeddings configuration
# LangChain automatically handles chunk batching behind the scenes!
embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

# 4. Manual loop batching
batch_size = 100
all_embeddings = []

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]
    print(f"Processing batch {i // batch_size + 1}: items {i} to {i + len(batch_texts)}")
    
    # Generate embeddings just for this specific batch
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    
    # Accumulate results
    all_embeddings.extend(batch_embeddings)

print(f"Successfully generated {len(all_embeddings)} embeddings.")
print(f"Sample of first embedding vector: {all_embeddings[0][:5]}...")

### Vectorstores, Chroma

In [ ]:
# Fazer o mesmo com ChromaDb
OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)
#texts = [chunk.page_content for chunk in chunks]

# 3. Native LangChain Embeddings configuration
# LangChain automatically handles chunk batching behind the scenes!
embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

docsearch = Chroma.from_documents(chunks, embeddings_model)
query = "Langchain"
docs = docsearch.similarity_search(query)
print(docs[0].page_content)

### Vector store-backed retrievers

In [ ]:
# Fazer o mesmo com ChromaDb
OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)

# 3. Native LangChain Embeddings configuration
# LangChain automatically handles chunk batching behind the scenes!
embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

docsearch = Chroma.from_documents(chunks, embeddings_model)

retriever = docsearch.as_retriever()
docs = retriever.invoke("Langchain")

for i, doc in enumerate(docs):
    print(f"\n--- Retrieved Document {i+1} ---")
    print(f"Content (first 300 chars): {doc.page_content[:300]}...")
    print(f"Metadata: {json.dumps(doc.metadata, indent=4)}")    


### Parent Document Retrievers

In [ ]:
OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# Set up two different text splitters for a hierarchical splitting approach:

# 1. Parent splitter creates larger chunks (2000 characters)
# This is used to split documents into larger, more contextually complete sections
parent_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=20, separator='\n')

# 2. Child splitter creates smaller chunks (400 characters)
# This is used to split the parent chunks into smaller pieces for more precise retrieval
child_splitter = CharacterTextSplitter(chunk_size=400, chunk_overlap=20, separator='\n')

embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

# Create a Chroma vector store with:
# - A specific collection name "split_parents" for organization
# - The previously configured Watson embeddings function
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=embeddings_model
)

# Set up an in-memory storage layer for the parent documents
# This will store the larger chunks that provide context, but won't be directly embedded
store = InMemoryStore()

# Create a ParentDocumentRetriever instance that implements hierarchical document retrieval
retriever = ParentDocumentRetriever(
    # The vector store where child document embeddings will be stored and searched
    # This Chroma instance will contain the embeddings for the smaller chunks
    vectorstore=vectorstore,
    
    # The document store where parent documents will be stored
    # These larger chunks won't be embedded but will be retrieved by ID when needed
    docstore=store,
    
    # The splitter used to create small chunks (400 chars) for precise vector search
    # These smaller chunks are embedded and used for similarity matching
    child_splitter=child_splitter,
    
    # The splitter used to create larger chunks (2000 chars) for better context
    # These parent chunks provide more complete information when retrieved
    parent_splitter=parent_splitter,
)

retriever.add_documents(document)

len(list(store.yield_keys()))
sub_docs = vectorstore.similarity_search("Langchain")
print(sub_docs[0].page_content)
retrieved_docs = retriever.invoke("Langchain")
print(retrieved_docs[0].page_content)

### Retrieval QA

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

OPENROUTER_MODEL = "openai/text-embedding-3-small"

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

# 2. Smart, larger chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       
    chunk_overlap=120,    
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(document)

embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

docsearch = Chroma.from_documents(chunks, embeddings_model)

# Create a RetrievalQA chain by configuring:
qa = RetrievalQA.from_chain_type(
    # The language model to use for generating answers
    llm=model,
    
    # The chain type "stuff" means all retrieved documents are simply concatenated and passed to the LLM
    chain_type="stuff",
    
    # The retriever component that will fetch relevant documents
    # docsearch.as_retriever() converts the vector store into a retriever interface
    retriever=docsearch.as_retriever(),
    
    # Whether to include the source documents in the response
    # Set to False to return only the generated answer
    return_source_documents=False
)

# Define a query to test the QA system
# This question asks about the main topic of the paper
query = "what is this paper discussing?"

# Execute the QA chain with the query
# This will:
# 1. Send the query to the retriever to get relevant documents
# 2. Combine those documents using the "stuff" method
# 3. Send the query and combined documents to the Llama LLM
# 4. Return the generated answer (without source documents)
qa.invoke(query)


### **Building a Simple Retrieval System with LangChain**

In [ ]:
# 1. Load a document about AI
loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
documents = loader.load()

# 2. Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# 3. Set up the embedding model
OPENROUTER_MODEL = "openai/text-embedding-3-small"

embeddings_model = OpenAIEmbeddings(
    model=OPENROUTER_MODEL,
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base=OPENROUTER_DEFAULT_BASE_URL,
    dimensions=1024
)

# 4. Create a vector store
vector_store = Chroma.from_documents(chunks, embeddings_model)

# 5. Create a retriever from the vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 6. Define a function to search for relevant information
def search_documents(query, top_k=3):
    """Search for documents relevant to a query"""
    # Use the retriever to get relevant documents
    docs = retriever.invoke(query)
    
    # Limit to top_k if specified
    return docs[:top_k]

# 7. Test with a few queries
test_queries = [
    "What is LangChain?",
    "How do retrievers work?",
    "Why is document splitting important?"
]


for query in test_queries:
    print(f"\nQuery: {query}")
    results = search_documents(query)
    
    # Print the results
    print(f"Found {len(results)} relevant documents:")
    for i, doc in enumerate(results):
        print(f"\nResult {i+1}: {doc.page_content[:150]}...")
        print(f"Source: {doc.metadata.get('source', 'Unknown')}")



## Memory

#### Chat Message History

In [42]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a new conversation history object
# This will store the back-and-forth messages in the conversation
history = ChatMessageHistory()

# Add an initial greeting message from the AI to the history
# This represents a message that would have been sent by the AI assistant
history.add_ai_message("hi!")

# Add a user's question to the conversation history
# This represents a message sent by the user
history.add_user_message("what is the capital of France?")

for message in history.messages:
    print(f"{message.type}: {message.content}")


ai_response = model.invoke(history.messages)
print("AI Response: ", ai_response.content)
print(json.dumps(ai_response.response_metadata, indent=4))


ai: hi!
human: what is the capital of France?
AI Response:  The capital of France is Paris.
{
    "model_name": "openai/gpt-4o-mini",
    "id": "gen-1780005356-pkmeL9gVvlaty1DKhg3n",
    "created": 1780005356,
    "object": "chat.completion",
    "finish_reason": "stop",
    "logprobs": null,
    "model_provider": "openrouter",
    "system_fingerprint": "fp_54f26dc974"
}


#### Conversation Buffer

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

conversation = ConversationChain(
    # The language model to use for generating responses
    llm=model,
    
    # Set verbose to True to see the full prompt sent to the LLM, including memory contents
    verbose=True,
    
    # Initialize with ConversationBufferMemory that will:
    # - Store all conversation turns (user inputs and AI responses)
    # - Append the entire conversation history to each new prompt
    # - Provide context for the LLM to generate contextually relevant responses
    memory=ConversationBufferMemory()
)

reply = conversation.invoke(input="Hello, I am a little cat. Who are you?")
print("Human: ", reply['input'])
print("AI: ", reply['response'])
reply = conversation.invoke(input="What can you do?")
print("Human: ", reply['input'])
print("AI: ", reply['response'])
reply = conversation.invoke(input="Who am I?.")
print("Human: ", reply['input'])
print("AI: ", reply['response'])

#### **Building a Chatbot with Memory using LangChain**


In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# 2. Create a simple conversation with chat history
history = ChatMessageHistory()

# Add some initial messages
history.add_user_message("Hello, my name is Alice.")
history.add_ai_message("Hello Alice! It's nice to meet you. How can I help you today?")

# 3. Print the current conversation history
# print("Initial Chat History:")
for message in history.messages:
    sender = "Human" if isinstance(message, HumanMessage) else "AI"
    print(f"{sender}: {message.content}")

# 4. Set up a conversation chain with memory
memory = ConversationBufferMemory(chat_memory=history)
conversation = ConversationChain(
    llm=model,
    memory=memory,
    verbose=False #True to see the full prompt with memory contents at each turn
)

# 5. Function to simulate a conversation
def chat_simulation(conversation, inputs):
    """Run a series of inputs through the conversation chain and display responses"""
    print("\n=== Beginning Chat Simulation ===")
    
    for i, user_input in enumerate(inputs):
        print(f"\n--- Turn {i+1} ---")
        print(f"Human: {user_input}")
        
        # Get response from the conversation chain
        response = conversation.invoke(input=user_input)
        
        # Print the AI's response
        print(f"AI: {response['response']}")
    
    print("\n=== End of Chat Simulation ===")

# 6. Test with a series of related questions
test_inputs = [
    "My favorite color is blue.",
    "I enjoy hiking in the mountains.",
    "What activities would you recommend for me?",
    "What was my favorite color again?",
    "Can you remember both my name and my favorite color?"
]

chat_simulation(conversation, test_inputs)

# 7. Examine the conversation memory
print("\nFinal Memory Contents:")
print(conversation.memory.buffer)

# 8. Create a new conversation with a different type of memory (optional)
# Create a summarizing memory that will compress the conversation
summary_memory = ConversationSummaryMemory(llm=model)
# Save the initial context to the summary memory
summary_memory.save_context(
    {"input": "Hello, my name is Alice."}, 
    {"output": "Hello Alice! It's nice to meet you. How can I help you today?"}
)
summary_conversation = ConversationChain(
   llm=model,
   memory=summary_memory,
   verbose=False #True to see the full prompt with memory contents at each turn
)


print("\\\\\\\\n\\n=== Testing Conversation Summary Memory ===")
# Let's use the same inputs for comparison
chat_simulation(summary_conversation, test_inputs)

print("\\nFinal Summary Memory Contents:")
print(summary_memory.buffer)

# 9. Compare the two memory types
print("\n=== Memory Comparison ===")
print(f"Buffer Memory Size: {len(conversation.memory.buffer)} characters")
print(f"Summary Memory Size: {len(summary_memory.buffer)} characters")
print("\nThe conversation summary memory typically creates a more compact representation of the chat history.")

## Chains

### Simple Chains

#### Traditional Approach: LLMChain

In [24]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create a template string for generating recommendations of classic dishes from a given location
# The template includes:
# - Instructions for the task (recommending a classic dish)
# - A placeholder {location} that will be replaced with user input
# - A format indicator for the expected response
template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}
 YOUR RESPONSE:
"""

# Create a PromptTemplate object by providing:
# - The template string defined above
# - A list of input variables that will be used to format the template
prompt_template = PromptTemplate(template=template, input_variables=['location'])

# Create an LLMChain that connects:
# - The Llama language model (llama_llm)
# - The prompt template configured for location-based dish recommendations
# - An output_key 'meal' that specifies the key name for the chain's response in the output dictionary
location_chain = LLMChain(llm=model, prompt=prompt_template, output_key='meal')

# Invoke the chain with 'China' as the location input
# This will:
# 1. Format the template with {location: 'China'}
# 2. Send the formatted prompt to the Llama LLM
# 3. Return a dictionary with the response under the key 'meal'
response = location_chain.invoke(input={'location':'China'})

print(response)


{'location': 'China', 'meal': "One classic dish from China is Peking Duck. This renowned dish hails from Beijing and is famous for its crispy skin and tender meat. Traditionally, the duck is seasoned, air-dried, and roasted until the skin is perfectly crispy. It is typically served with thin pancakes, hoisin sauce, and sliced scallions, allowing diners to create their own wraps. Peking Duck is not just a meal; it's a celebrated culinary experience!"}


#### Modern Approach: LCEL

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}
 YOUR RESPONSE:
"""

# Create a prompt template using the from_template method
prompt = PromptTemplate.from_template(template)

# Create a chain using LangChain Expression Language (LCEL) with the pipe operator
# This creates a processing pipeline that:
# 1. Formats the prompt with the input values
# 2. Sends the formatted prompt to the Llama LLM
# 3. Parses the output to extract just the string response
location_chain_lcel = prompt | model | StrOutputParser()

# Invoke the chain with 'China' as the location
result = location_chain_lcel.invoke({"location": "China"})

# Print the result (the recommended classic dish from China)
print(result)

#### Traditional Sequencial Chain

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

template = """Your job is to come up with a classic dish from the area that the users suggests. Just the DISH, no explanation.
{location}
 YOUR RESPONSE:
"""

input_location = "Portugal"

# Create a prompt template using the from_template method
prompt_template = PromptTemplate(template=template, input_variables=['location'])

# Invoke the chain with input_location as the location
meal_chain = LLMChain(llm=model, prompt=prompt_template, output_key='meal')
meal_response = meal_chain.invoke(input={'location': input_location})
#print output key 'meal' from the response
print(f"Classic Dish: {meal_response['meal']}")

# Create a template for generating a recipe based on a meal
template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home. Just the RECIPE, no explanation.
 YOUR RESPONSE:
"""

# Create a PromptTemplate with 'meal' as the input variable
prompt_template = PromptTemplate(template=template, input_variables=['meal'])

# Create an LLMChain (chain 2) for generating recipes
# The output_key='recipe' defines how this chain's output will be referenced in later chains
dish_chain = LLMChain(llm=model, prompt=prompt_template, output_key='recipe')
dish_response = dish_chain.invoke(input={'meal': meal_response['meal']})

#Print output key 'recipe' from the response
print(f"Recipe: {dish_response['recipe']}")

# Create a template for estimating cooking time based on a recipe
# This template asks the LLM to analyze a recipe and estimate preparation time
template = """Given the recipe {recipe}, estimate how much time I need to cook it. Just the TIME, no explanation.
 YOUR RESPONSE:
"""

# Create a PromptTemplate with 'recipe' as the input variable
prompt_template = PromptTemplate(template=template, input_variables=['recipe'])

# Create an LLMChain (chain 3) for estimating cooking time
# The output_key='time' defines the key for this chain's output in the final result
recipe_chain = LLMChain(llm=model, prompt=prompt_template, output_key='time')
recipe_response = recipe_chain.invoke(input={'recipe': dish_response['recipe']})

#Print output key 'time' from the response
print(f"Estimated Cooking Time: {recipe_response['time']}")

### Create a sequential chain that connects the three individual chains together
overall_chain = SequentialChain(chains=[meal_chain, dish_chain, recipe_chain], input_variables=['location'], output_variables=['meal', 'recipe', 'time'])
overall_response = overall_chain.invoke({'location': input_location})
print("\n---")
print(json.dumps(overall_response, indent=4))

#### Modern Approach: LCEL

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Define the templates for each step
location_template = """Your job is to come up with a classic dish from the area that the users suggests.
{location}

YOUR RESPONSE:
"""

dish_template = """Given a meal {meal}, give a short and simple recipe on how to make that dish at home.

YOUR RESPONSE:
"""

time_template = """Given the recipe {recipe}, estimate how much time I need to cook it.

YOUR RESPONSE:
"""

# Create the location chain using LCEL (LangChain Expression Language)
# This chain takes a location and returns a classic dish from that region
location_chain_lcel = (
    PromptTemplate.from_template(location_template)  # Format the prompt with location
    | model                                    # Send to the LLM
    | StrOutputParser()                              # Extract the string response
)

# Create the dish chain using LCEL
# This chain takes a meal name and returns a recipe
dish_chain_lcel = (
    PromptTemplate.from_template(dish_template)      # Format the prompt with meal
    | model                                    # Send to the LLM
    | StrOutputParser()                              # Extract the string response
)

# Create the time estimation chain using LCEL
# This chain takes a recipe and returns an estimated cooking time
time_chain_lcel = (
    PromptTemplate.from_template(time_template)      # Format the prompt with recipe
    | model                                    # Send to the LLM
    | StrOutputParser()                              # Extract the string response
)

# Combine all chains into a single workflow using RunnablePassthrough.assign
# RunnablePassthrough.assign adds new keys to the input dictionary without removing existing ones
overall_chain_lcel = (
    # Step 1: Generate a meal based on location and add it to the input dictionary
    RunnablePassthrough.assign(meal=lambda x: location_chain_lcel.invoke({"location": x["location"]}))
    # Step 2: Generate a recipe based on the meal and add it to the input dictionary
    | RunnablePassthrough.assign(recipe=lambda x: dish_chain_lcel.invoke({"meal": x["meal"]}))
    # Step 3: Estimate cooking time based on the recipe and add it to the input dictionary
    | RunnablePassthrough.assign(time=lambda x: time_chain_lcel.invoke({"recipe": x["recipe"]}))
)
# Run the chain
result = overall_chain_lcel.invoke({"location": "China"})
print(json.dumps(result, indent=4))

#### Implementing Multi-Step Processing with Different Chain Approaches

In [ ]:
OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Sample product reviews for testing
positive_review = """I absolutely love this coffee maker! It brews quickly and the coffee tastes amazing. 
The built-in grinder saves me so much time in the morning, and the programmable timer means 
I wake up to fresh coffee every day. Worth every penny and highly recommended to any coffee enthusiast."""

negative_review = """Disappointed with this laptop. It's constantly overheating after just 30 minutes of use, 
and the battery life is nowhere near the 8 hours advertised - I barely get 3 hours. 
The keyboard has already started sticking on several keys after just two weeks. Would not recommend to anyone."""

# Step 1: Define the prompt templates for each processing step
sentiment_template = """Analyze the sentiment of the following product review as positive, negative, or neutral.
Provide your analysis in the format: "SENTIMENT: [positive/negative/neutral]"
Review: {review}
Your analysis:
"""

summary_template = """Summarize the following product review into 3-5 key bullet points.
Each bullet point should be concise and capture an important aspect mentioned in the review.
Review: {review}
Sentiment: {sentiment}
Key points:
"""

response_template = """Write a helpful response to a customer based on their product review.
If the sentiment is positive, thank them for their feedback. If negative, express understanding 
and suggest a solution or next steps. Personalize based on the specific points they mentioned.
Review: {review}
Sentiment: {sentiment}
Key points: {summary}
Response to customer:
"""

# PART 1: Traditional Chain Approach
# Create prompt templates for each step
sentiment_prompt = PromptTemplate.from_template(sentiment_template)
summary_prompt = PromptTemplate.from_template(summary_template)
response_prompt = PromptTemplate.from_template(response_template)

# Create LLMChains for each step
sentiment_chain = LLMChain(llm=model, prompt=sentiment_prompt, output_key='sentiment')
summary_chain = LLMChain(llm=model, prompt=summary_prompt, output_key='summary')
response_chain = LLMChain(llm=model, prompt=response_prompt, output_key='response')

# Create a SequentialChain to connect all steps
traditional_chain = SequentialChain(
    chains=[sentiment_chain, summary_chain, response_chain],
    input_variables=["review"],
    output_variables=["sentiment", "summary", "response"],
    verbose=True
)

# PART 2: LCEL Approach
# Create individual chain components using the pipe operator (|)
sentiment_chain_lcel = sentiment_prompt | model | StrOutputParser()
summary_chain_lcel = summary_prompt | model | StrOutputParser()
response_chain_lcel = response_prompt | model | StrOutputParser()

# Connect the components using RunnablePassthrough.assign()
overall_chain_lcel = (
    RunnablePassthrough.assign(sentiment=lambda x: sentiment_chain_lcel.invoke({"review": x["review"]}))
    | RunnablePassthrough.assign(summary=lambda x: summary_chain_lcel.invoke({"review": x["review"], "sentiment": x["sentiment"]}))
    | RunnablePassthrough.assign(response=lambda x: response_chain_lcel.invoke({"review": x["review"], "sentiment": x["sentiment"], "summary": x["summary"]}))
)


# Test both implementations
def test_chains(review):
    """Test both chain implementations with the given review"""
    print("\n" + "="*50)
    print(f"TESTING WITH REVIEW:\n{review[:100]}...\n")
    
    print("TRADITIONAL CHAIN RESULTS:")
    traditional_results = traditional_chain.invoke({"review": review})
    print(f"Sentiment: {traditional_results['sentiment']}")
    print(f"Summary: {traditional_results['summary']}")
    print(f"Response: {traditional_results['response']}")
    
    print("\nLCEL CHAIN RESULTS:")
    lcel_results = overall_chain_lcel.invoke({"review": review})
    print(f"Sentiment: {lcel_results['sentiment']}")
    print(f"Summary: {lcel_results['summary']}")
    print(f"Response: {lcel_results['response']}")
    
    print("="*50)

# Run tests on both a positive and negative review
test_chains(positive_review)
test_chains(negative_review)

### Tools and Agents